# Notebook 03 — Evaluate All Experiments

**Member responsible: C**

Runs evaluation for every experiment and saves results CSVs to Drive.
Run this notebook once per experiment by updating `eval.yaml`.

| Experiment | eval.yaml: checkpoint_path | Benchmark |
|---|---|---|
| Exp 0 zero-shot      | null                                    | GSM8K + ARC |
| Exp 1 few-shot CoT   | null (uses cot_prompting.py)            | GSM8K + ARC |
| Exp 2 adaptation     | results/checkpoints/adaptation/         | GSM8K |
| Exp 3 distillation   | results/checkpoints/distillation/       | GSM8K |
| Exp 4 filtered       | results/checkpoints/filtered/           | GSM8K |
| Exp 5 cross-task     | results/checkpoints/filtered/ (same)   | ARC only |

In [ ]:
# Step 1: Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

# TODO: replace with your actual GitHub repo URL
!git clone https://github.com/YOUR_USERNAME/SC4001-Group133.git /content/SC4001-Group133
%cd /content/SC4001-Group133
!pip install -r requirements.txt -q

In [ ]:
# Step 2: Copy checkpoint from Drive (skip for Exp 0/1)
# TODO: update CHECKPOINT_NAME to match the run you want to evaluate
CHECKPOINT_NAME = 'adaptation'  # e.g. 'adaptation', 'distillation', 'filtered', 'lora_r4'
!mkdir -p results/checkpoints
!cp -r /content/drive/MyDrive/SC4001/results/checkpoints/{CHECKPOINT_NAME} results/checkpoints/
print(f'Checkpoint loaded: {CHECKPOINT_NAME}')

In [ ]:
# Step 3: Run Exp 0 — Zero-shot baseline (no checkpoint needed)
from src.baselines.cot_prompting import run_baseline
from src.evaluation.metrics import compute_summary_stats, save_results

results_exp0 = run_baseline(mode='zero_shot', dataset_name='gsm8k')
save_results(results_exp0, 'results/exp0_zero_shot_gsm8k.csv')
print('Exp 0 done:', compute_summary_stats(results_exp0))

In [ ]:
# Step 4: Run Exp 1 — Few-shot CoT baseline
results_exp1 = run_baseline(mode='few_shot', dataset_name='gsm8k')
save_results(results_exp1, 'results/exp1_few_shot_gsm8k.csv')
print('Exp 1 done:', compute_summary_stats(results_exp1))

In [ ]:
# Step 5: Run Exp 2-5 — update eval.yaml checkpoint_path, then run this cell
# TODO: edit configs/eval.yaml to set the right checkpoint_path and experiment name
EXPERIMENT_NAME = 'exp2_adaptation_r16'  # change for each run

from src.evaluation.eval_gsm8k import evaluate_gsm8k
results = evaluate_gsm8k(experiment_name=EXPERIMENT_NAME)
print(f'{EXPERIMENT_NAME} GSM8K done.')

In [ ]:
# Step 6: Run Exp 5 (D3) — ARC cross-task transfer evaluation
# Uses the Exp 4 (filtered distillation) checkpoint — set checkpoint_path in eval.yaml
from src.evaluation.eval_arc import evaluate_arc
results_arc = evaluate_arc(experiment_name='exp5_d3_cross_task')
print('Exp 5 ARC done.')

In [ ]:
# Step 7: Save all results to Drive
!cp -r results/ /content/drive/MyDrive/SC4001/results/
print('All results saved to Drive.')